In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral"],
    "mathtext.fontset": "stix",

    # Important for clean PDF/Illustrator rendering
    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    # Avoid weird minus signs in some PDF viewers
    "axes.unicode_minus": False,
})

model_name = "gemma12b"
RESULT_NAME = "combined"
CUSTOM_RESULT_NAME = None  # Set to e.g. "answer_colon" if a custom run has multiple sites.

# New layout: one JSON produced with run_head_interchange_intervention.py --role-prompt-scores.
def first_existing_path(paths):
    for path in paths:
        if Path(path).exists() or (Path("binding-iclr") / path).exists():
            return path
    return paths[0]


USE_ROLE_PROMPT_SCORES = True
ROLE_PROMPT_SCORES_PATHS = [
    # Path(f"./results/path_patching/{model_name}/head_interchange_with_c_to_a_milk.json"),
    Path(f"./results/path_patching/{model_name}/out_all_v2.json"),
    # Path(f"./results/path_patching/{model_name}/out_all.json"),
]
ROLE_PROMPT_SCORES_PATH = first_existing_path(ROLE_PROMPT_SCORES_PATHS)

# Keep (C) and (D) as before if the new role_prompt_scores JSON skips them.
FILL_SKIPPED_ROLES_FROM_LEGACY = True

ROLE_ORDER = [
    "value_fetcher",
    "pos_transmitter",
    "pos_detector",
    "struct_reader_swap_target",
]

EXCLUDED_ROLE_KEYS = {
    "struct_reader_context_target_box_to_target_box",
}

ROLE_LABELS = {
    "value_fetcher": "(A) Answer retriever",
    "pos_transmitter": "(B) Dereferencer",
    "pos_detector": "(C) Position updater",
    "struct_reader_swap_target": "(D) Position transmitter",
}

# Old layout fallback: one JSON per role.
LEGACY_RUNS = {
    "value_fetcher": Path(f"./results/path_patching/{model_name}/head_interchange_value_fetcher4.json"),
    "pos_transmitter": Path(f"./results/path_patching/{model_name}/head_interchange_pos_transmitter3.json"),
    "pos_detector": Path(f"./results/path_patching/{model_name}/head_interchange_pos_detector.json"),
    "struct_reader_swap_target": Path(f"./results/path_patching/{model_name}/head_interchange_struct_reader_swap_target3.json"),
}

# Original score: logit(label after patch) - logit(label before patch).
# Payload uses source_label; pointer keeps the old query_object score.
SCORE_SPECS = {
    "payload": (
        "payload_prompt",
        "patched_minus_original_mean_source_label_logit",
    ),
    "pointer": (
        "pointer_prompt",
        "patched_minus_original_mean_query_object_logit",
    ),
}

# Same fields for the old per-role JSON layout.
DELTA_FIELDS = {
    "payload": "patched_minus_original_mean_source_label_logit",
    "pointer": "patched_minus_original_mean_query_object_logit",
}

POINTER_CUSTOM_EXPERIMENT = (
    "c_to_a_milk",
    "patched_minus_original_mean_query_object_logit",
)

METRIC_ORDER = ["payload", "pointer"]

METRIC_TICK_LABELS = {
    "payload": "control",
    "pointer": "pointer",
    # "pointer2": "pointer2\n(C->A milk)",
}

ROLE_COLORS = {
    "value_fetcher": "#0076BA",
    "pos_transmitter": "#00766D",
    "pos_detector": "#F27200",
    "struct_reader_swap_target": "#D43F00",
}


def resolve_result_path(path):
    path = Path(path)
    if path.exists():
        return path
    candidate = Path("binding-iclr") / path
    if candidate.exists():
        return candidate
    raise FileNotFoundError(path)


def select_result_block(results, preferred=RESULT_NAME):
    if preferred in results:
        return preferred, results[preferred]
    if len(results) == 1:
        return next(iter(results.items()))
    names = ", ".join(results)
    raise KeyError(f"Could not find result {preferred!r}; available results: {names}")


def select_custom_result_block(results, preferred=None, fallback_sites=None):
    candidates = []
    if CUSTOM_RESULT_NAME is not None:
        candidates.append(CUSTOM_RESULT_NAME)
    if preferred is not None:
        candidates.append(preferred)
    candidates.append(RESULT_NAME)
    candidates.extend(fallback_sites or [])

    for candidate in candidates:
        if candidate in results:
            return candidate, results[candidate]
    if len(results) == 1:
        return next(iter(results.items()))
    return None, None


def load_c_to_a_milk_pointer_value(payload, preferred_result=None, fallback_sites=None):
    custom_experiments = payload.get("custom_experiments") or {}
    experiment_name, delta_field = POINTER_CUSTOM_EXPERIMENT
    experiment = custom_experiments.get(experiment_name)
    if not experiment:
        return None, None
    result_name, result_block = select_custom_result_block(
        experiment.get("results", {}),
        preferred=preferred_result,
        fallback_sites=fallback_sites,
    )
    if result_block is None:
        return None, None
    deltas = result_block.get("deltas", {})
    if delta_field not in deltas:
        raise KeyError(
            f"custom_experiments/{experiment_name}/{result_name} is missing {delta_field!r}"
        )
    return deltas[delta_field], result_name


def load_role_prompt_values(role_key, role_block, top_level_payload=None, use_top_level_custom=False):
    prompt_scores = role_block.get("prompt_scores", {})
    values = {}
    result_names = {}
    first_result_name = None
    for metric_label, (prompt_name, delta_field) in SCORE_SPECS.items():
        if prompt_name not in prompt_scores:
            raise KeyError(f"{role_key} is missing prompt score {prompt_name!r}")
        result_name, result_block = select_result_block(prompt_scores[prompt_name].get("results", {}))
        first_result_name = first_result_name or result_name
        deltas = result_block.get("deltas", {})
        if delta_field not in deltas:
            raise KeyError(f"{role_key}/{prompt_name}/{result_name} is missing {delta_field!r}")
        values[metric_label] = deltas[delta_field]
        result_names[metric_label] = result_name

    pointer_value, pointer_result_name = load_c_to_a_milk_pointer_value(
        role_block,
        preferred_result=first_result_name,
        fallback_sites=role_block.get("active_sites"),
    )
    if pointer_value is None and use_top_level_custom and top_level_payload is not None:
        pointer_value, pointer_result_name = load_c_to_a_milk_pointer_value(
            top_level_payload,
            preferred_result=first_result_name,
            fallback_sites=role_block.get("active_sites"),
        )
    if pointer_value is not None:
        values["pointer2"] = pointer_value
        result_names["pointer2"] = pointer_result_name
    return values, result_names


def load_legacy_values(role_key):
    input_path = resolve_result_path(LEGACY_RUNS[role_key])
    payload = json.loads(input_path.read_text())
    result_name, result_block = select_result_block(payload.get("results", {}))
    deltas = result_block.get("deltas", {})
    values = {label: deltas[field] for label, field in DELTA_FIELDS.items()}
    result_names = {label: result_name for label in values}
    pointer_value, pointer_result_name = load_c_to_a_milk_pointer_value(
        payload,
        preferred_result=result_name,
    )
    if pointer_value is not None:
        values["pointer2"] = pointer_value
        result_names["pointer2"] = pointer_result_name
    return values, result_names, input_path


def make_row(role_key, values, result_names, source, active_sites=None, hook_names=None, input_path=None):
    return {
        "role_key": role_key,
        "role": ROLE_LABELS.get(role_key, role_key),
        "values": values,
        "result_names": result_names,
        "source": source,
        "active_sites": active_sites,
        "hook_names": hook_names,
        "input_path": str(input_path) if input_path is not None else None,
    }


def load_role_prompt_score_rows(path):
    input_path = resolve_result_path(path)
    payload = json.loads(input_path.read_text())
    role_prompt_scores = payload.get("role_prompt_scores")
    if not role_prompt_scores:
        raise KeyError(
            f"{input_path} has no role_prompt_scores block. "
            "Rerun with --role-prompt-scores or set USE_ROLE_PROMPT_SCORES = False."
        )

    available_roles = [role for role in role_prompt_scores if role not in EXCLUDED_ROLE_KEYS]
    ordered_roles = [
        role
        for role in ROLE_ORDER
        if role not in EXCLUDED_ROLE_KEYS and (role in role_prompt_scores or role in LEGACY_RUNS)
    ]
    ordered_roles += sorted(role for role in available_roles if role not in ordered_roles)
    active_role_count = sum(
        1
        for role_key, role_block in role_prompt_scores.items()
        if role_key not in EXCLUDED_ROLE_KEYS
        if role_block and not role_block.get("skipped")
    )

    rows = []
    skipped = {}
    for role_key in ordered_roles:
        role_block = role_prompt_scores.get(role_key)
        use_legacy = role_block is None or role_block.get("skipped")
        if use_legacy:
            reason = "missing from role_prompt_scores" if role_block is None else role_block.get("skip_reason", "skipped")
            if FILL_SKIPPED_ROLES_FROM_LEGACY and role_key in LEGACY_RUNS:
                values, result_names, legacy_path = load_legacy_values(role_key)
                rows.append(make_row(role_key, values, result_names, "legacy", input_path=legacy_path))
            else:
                skipped[role_key] = reason
            continue

        values, result_names = load_role_prompt_values(
            role_key,
            role_block,
            top_level_payload=payload,
            use_top_level_custom=True,
        )
        rows.append(make_row(
            role_key,
            values,
            result_names,
            "role_prompt_scores",
            active_sites=role_block.get("active_sites"),
            hook_names=role_block.get("hook_names"),
            input_path=input_path,
        ))

    return rows, skipped, input_path, payload


def load_legacy_rows():
    rows = []
    first_path = None
    for role_key in ROLE_ORDER:
        values, result_names, input_path = load_legacy_values(role_key)
        first_path = first_path or input_path
        rows.append(make_row(role_key, values, result_names, "legacy", input_path=input_path))
    return rows, {}, first_path, None


if USE_ROLE_PROMPT_SCORES:
    rows, skipped_roles, input_path, payload = load_role_prompt_score_rows(ROLE_PROMPT_SCORES_PATH)
else:
    rows, skipped_roles, input_path, payload = load_legacy_rows()

metric_labels = [
    label
    for label in METRIC_ORDER
    if any(row["values"].get(label) is not None for row in rows)
]
rows


In [ ]:
from matplotlib.patches import Patch

if not rows:
    raise ValueError("No plottable rows were loaded.")
if not metric_labels:
    raise ValueError("No plottable metrics were loaded.")

fig, ax = plt.subplots(figsize=(7.5, 6))
group_width = 0.72
bar_width = group_width / max(len(metric_labels), 1)

for role_idx, row in enumerate(rows):
    color = ROLE_COLORS.get(row["role_key"], "#666666")
    for metric_idx, metric_label in enumerate(metric_labels):
        value = row["values"].get(metric_label)
        if value is None:
            continue
        offset = (metric_idx - (len(metric_labels) - 1) / 2) * bar_width
        is_control = metric_label == "payload"
        bars = ax.bar(
            [role_idx + offset],
            [value],
            width=bar_width * 0.9,
            facecolor="none" if is_control else color,
            edgecolor=color,
            hatch="///" if is_control else None,
            linewidth=0.9 if is_control else 0.0,
        )
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=18)

ax.axhline(0, color="gray", linestyle="--", linewidth=1.5)
ax.set_ylabel(r"$\Delta$ logit (patched - original)", fontsize=20)
ax.tick_params(axis="y", labelsize=20)
ax.set_xticks(range(len(rows)))
ax.set_xticklabels([row["role"].split(") ")[0] + ")" for row in rows], fontsize=20)
# ax.set_title(f"{model_name}: head-role logit deltas")
role_handles = [
    Patch(
        facecolor=ROLE_COLORS.get(row["role_key"], "#666666"),
        edgecolor=ROLE_COLORS.get(row["role_key"], "#666666"),
        label=row["role"],
    )
    for row in rows
]
metric_handles = [
    Patch(facecolor="none", edgecolor="black", hatch="///", label="control"),
    Patch(facecolor="#888888", edgecolor="#888888", label="pointer"),
]
ax.legend(handles=role_handles, frameon=True, fontsize=18, loc='upper right')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
